# Production Pipeline & Optimization (Days 8-9)

## Objective
Build production-ready inference pipeline and optimize H2 production timing.

## Components
1. **Batch Inference**: Score all hours with high-price predictions
2. **Production Optimizer**: Recommend optimal H2 production windows
3. **Gold Layer**: Save predictions and recommendations
4. **Query Optimization**: Cache tables, explain plans, optimize queries

## Business Value
* **Cost Savings**: Avoid producing H2 during expensive hours
* **Green Energy**: Maximize production during high renewable periods
* **Risk Management**: Predict price spikes in advance
* **Scalability**: Production-grade pipeline with monitoring

## Output Tables
* `h2_gold_price_predictions`: Hourly price risk scores
* `h2_gold_optimal_production_windows`: Recommended production times
* `h2_ops_pipeline_log`: Pipeline execution tracking

# Production Pipeline & Optimization (Days 8-9)

## Objective
Build production-ready inference pipeline and optimize H2 production timing.

## Components
1. **Batch Inference**: Score all hours with high-price predictions
2. **Production Optimizer**: Recommend optimal H2 production windows
3. **Gold Layer**: Save predictions and recommendations
4. **Query Optimization**: Cache tables, explain plans, optimize queries

## Business Value
* **Cost Savings**: Avoid producing H2 during expensive hours
* **Green Energy**: Maximize production during high renewable periods
* **Risk Management**: Predict price spikes in advance
* **Scalability**: Production-grade pipeline with monitoring

## Output Tables
* `h2_gold_price_predictions`: Hourly price risk scores
* `h2_gold_optimal_production_windows`: Recommended production times
* `h2_ops_pipeline_log`: Pipeline execution tracking

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import mlflow
import mlflow.spark
import pandas as pd
import time

print("✅ Imports loaded")

In [0]:
# Data configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772775399"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

# MLflow configuration
EXPERIMENT_NAME = "/Users/labuser13955035_1772775399@vocareum.com/H2_Price_Classification"

# Output tables
PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_price_predictions"
OPTIMAL_WINDOWS_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_optimal_production_windows"
PIPELINE_LOG_TABLE = f"{CATALOG}.{SCHEMA}.h2_ops_pipeline_log"

print(f"Source: {SOURCE_TABLE}")
print(f"Output predictions: {PREDICTIONS_TABLE}")
print(f"Output recommendations: {OPTIMAL_WINDOWS_TABLE}")
print(f"Pipeline log: {PIPELINE_LOG_TABLE}")

In [0]:
# Get best run from MLflow experiment
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])

# Sort by test_auc descending
best_run = runs.sort_values("metrics.test_auc", ascending=False).iloc[0]
best_run_id = best_run["run_id"]
best_model_type = best_run["tags.model_type"]
best_auc = best_run["metrics.test_auc"]

print(f"🏆 Loading best model from MLflow:")
print(f"   Model Type: {best_model_type}")
print(f"   Run ID: {best_run_id}")
print(f"   Test AUC: {best_auc:.4f}")

# Load model
model_uri = f"runs:/{best_run_id}/model"
model = mlflow.spark.load_model(model_uri)

print(f"\n✅ Model loaded successfully")

In [0]:
print("Running batch inference...")
print("="*80)

# Load all data
df = spark.table(SOURCE_TABLE)

print(f"Total records to score: {df.count():,}")

# Make predictions
start_time = time.time()
predictions = model.transform(df)
scoring_time = time.time() - start_time

print(f"✅ Scoring complete in {scoring_time:.2f} seconds")

# Extract probability and predictions
from pyspark.ml.functions import vector_to_array

predictions_clean = predictions.select(
    "event_time_utc",  # Fixed: was mtu_cet_cest
    "zone",  # Fixed: was area
    "price_eur_mwh",
    "renewable_generation_mw",
    "non_renewable_generation_mw",
    "renewable_share_pct",
    F.col("prediction").cast("int").alias("predicted_high_price"),
    vector_to_array(F.col("probability")).getItem(1).alias("price_risk_score")
).withColumn(
    "prediction_timestamp",
    F.current_timestamp()
).withColumn(
    "model_type",
    F.lit(best_model_type)
).withColumn(
    "run_id",
    F.lit(best_run_id)
)

print(f"\nSample predictions:")
display(predictions_clean.orderBy(F.desc("price_risk_score")).limit(10))

In [0]:
# Save predictions
print(f"Saving predictions to: {PREDICTIONS_TABLE}")

predictions_clean.write.mode("overwrite").saveAsTable(PREDICTIONS_TABLE)

record_count = spark.table(PREDICTIONS_TABLE).count()
print(f"✅ Predictions saved: {record_count:,} records")

# Show summary statistics
print(f"\nPrediction Summary:")
summary = predictions_clean.groupBy("predicted_high_price").agg(
    F.count("*").alias("count"),
    F.avg("price_risk_score").alias("avg_risk_score"),
    F.min("price_risk_score").alias("min_risk_score"),
    F.max("price_risk_score").alias("max_risk_score")
).orderBy("predicted_high_price")

display(summary)

In [0]:
print("Identifying optimal H2 production windows...")
print("="*80)

# Criteria for optimal production:
# 1. Low price risk score (< 0.3)
# 2. High renewable share (> 15%)
# 3. Sufficient total generation

optimal_windows = predictions_clean.filter(
    (F.col("price_risk_score") < 0.3) &
    (F.col("renewable_share_pct") > 15.0)
).withColumn(
    "production_score",
    # Higher score = better production window
    (1 - F.col("price_risk_score")) * (F.col("renewable_share_pct") / 100)
).withColumn(
    "recommendation",
    F.when(F.col("production_score") > 0.15, "HIGHLY_RECOMMENDED")
     .when(F.col("production_score") > 0.10, "RECOMMENDED")
     .otherwise("CONSIDER")
)

print(f"✅ Optimal windows identified: {optimal_windows.count():,} hours")

# Show top 20 recommended production times
print(f"\nTop 20 Optimal Production Windows:")
top_windows = optimal_windows.orderBy(F.desc("production_score")).limit(20)
display(top_windows.select(
    "event_time_utc",  # Fixed: was mtu_cet_cest
    "price_eur_mwh",
    "price_risk_score",
    "renewable_share_pct",
    "production_score",
    "recommendation"
))

In [0]:
# Save optimal production windows
print(f"Saving optimal windows to: {OPTIMAL_WINDOWS_TABLE}")

optimal_windows.write.mode("overwrite").saveAsTable(OPTIMAL_WINDOWS_TABLE)

window_count = spark.table(OPTIMAL_WINDOWS_TABLE).count()
print(f"✅ Optimal windows saved: {window_count:,} records")

# Summary by recommendation level
print(f"\nRecommendation Summary:")
rec_summary = optimal_windows.groupBy("recommendation").agg(
    F.count("*").alias("count"),
    F.avg("production_score").alias("avg_production_score"),
    F.avg("price_risk_score").alias("avg_price_risk"),
    F.avg("renewable_share_pct").alias("avg_renewable_pct")
).orderBy(F.desc("avg_production_score"))

display(rec_summary)

In [0]:
print("Query Optimization Techniques")
print("="*80)

# Example query: Average price risk by hour of day
query = f"""
SELECT 
    hour(event_time_utc) as hour_of_day,
    avg(price_risk_score) as avg_risk,
    count(*) as count
FROM {PREDICTIONS_TABLE}
GROUP BY hour(event_time_utc)
ORDER BY hour_of_day
"""

# 1. Show explain plan
print("\n1. EXPLAIN PLAN (before optimization)")
print("-" * 80)
spark.sql(f"EXPLAIN FORMATTED {query}").show(truncate=False)

# 2. Time without cache
print("\n2. QUERY WITHOUT CACHE")
print("-" * 80)
t0 = time.time()
result = spark.sql(query)
count = result.count()
t1 = time.time()
print(f"Runtime: {t1-t0:.2f} seconds")
print(f"Rows: {count}")

# 3. Cache table
print("\n3. CACHING TABLE")
print("-" * 80)
spark.sql(f"CACHE TABLE {PREDICTIONS_TABLE}")
print(f"✅ Table cached")

# 4. Time with cache
print("\n4. QUERY WITH CACHE")
print("-" * 80)
t2 = time.time()
result_cached = spark.sql(query)
count_cached = result_cached.count()
t3 = time.time()
print(f"Runtime: {t3-t2:.2f} seconds")
print(f"Rows: {count_cached}")
print(f"\n⚡ Speedup: {(t1-t0)/(t3-t2):.2f}x faster with cache")

In [0]:
# Create pipeline execution log
print("Logging pipeline execution...")
print("="*80)

# Create ops schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ops")

# Create log table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PIPELINE_LOG_TABLE} (
    run_timestamp TIMESTAMP,
    pipeline_stage STRING,
    status STRING,
    record_count BIGINT,
    model_type STRING,
    run_id STRING,
    execution_time_sec DOUBLE
) USING DELTA
""")

# Log this pipeline run
log_entries = [
    ("batch_inference", "success", record_count, best_model_type, best_run_id, scoring_time),
    ("optimal_windows", "success", window_count, best_model_type, best_run_id, 0.0)
]

log_df = spark.createDataFrame(log_entries, ["pipeline_stage", "status", "record_count", "model_type", "run_id", "execution_time_sec"])
log_df = log_df.withColumn("run_timestamp", F.current_timestamp())

log_df.write.mode("append").saveAsTable(PIPELINE_LOG_TABLE)

print(f"✅ Pipeline execution logged to: {PIPELINE_LOG_TABLE}")
print(f"\nRecent pipeline runs:")
display(spark.table(PIPELINE_LOG_TABLE).orderBy(F.desc("run_timestamp")).limit(10))

## ✅ Production Pipeline Complete!

### What We Built
1. **Batch Inference**: Scored all 17,535 hours with price risk predictions
2. **Optimal Windows**: Identified best H2 production times (low price + high renewable)
3. **Gold Tables**: Saved predictions and recommendations for downstream use
4. **Query Optimization**: Demonstrated caching and explain plans
5. **Pipeline Logging**: Tracked execution for monitoring

### Gold Layer Outputs
* `h2_gold_price_predictions`: Hourly price risk scores with model metadata
* `h2_gold_optimal_production_windows`: Recommended production times with scores
* `h2_ops_pipeline_log`: Pipeline execution history

### Business Impact
* **Cost Savings**: Avoid producing H2 during high-price periods (> 75th percentile)
* **Green Energy**: Maximize production when renewables > 15%
* **Risk Management**: Proactive alerts for price spikes

### Production Deployment Checklist
- [ ] Schedule pipeline to run hourly/daily
- [ ] Set up alerting for high price risk periods
- [ ] Monitor model performance drift
- [ ] Retrain model monthly with new data
- [ ] Integrate recommendations with H2 production scheduling system

### Key Metrics to Monitor
* **Model Performance**: AUC, F1 score over time
* **Prediction Accuracy**: Compare predicted vs actual high-price periods
* **Cost Savings**: EUR saved by following recommendations
* **Green Production**: % of H2 produced during high renewable periods